# **Partie 1: Assistant RAG (Retrieval)**

**Objectif** : Répondre aux questions des employés en se basant strictement sur la documentation de l'hôtel.

**Documents sources** (data/part-2) :
- Convention collective - services alimentaires.pdf
- Hébergement - Critères du service d'entretien ménager dans un hôtel 4 diamants.pdf
- Membres du comité SST.pdf
- Principes de vie internes.pdf
- Processus - Gestion des réservations.pdf
- politiques_hotel_de_la_promenade_document.pdf

**Phase de Retrieval** :
1. Extraction du texte des PDFs
2. Chunking (découpage en segments)
3. Création d'un index vectoriel (FAISS ou ChromaDB)
4. Tests de stratégies de prompting (zero-shot, few-shot, CoT)
5. Documentation des conversations test

## **Imports**

In [33]:
# Imports
import os
import pandas as pd
import numpy as np
from pathlib import Path
import pdfplumber
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import warnings
import json
import ollama

warnings.filterwarnings('ignore')

In [3]:
# Configurations
PROJET_ROOT = Path.cwd().parent.parent

DATA_PART2 = PROJET_ROOT / "data" / "part-2"
DATA_PROCESSED = PROJET_ROOT / "data" / "processed"

MODELS_RAG_DIR = PROJET_ROOT / "models" / "1-rag"
MODELS_RAG_DIR.mkdir(parents=True, exist_ok=True)

### **Choix Du Modèle D'Embedding Pour Le Retrieval**

La phase de *retrieval* (recherche des documents pertinents) repose sur des modèles d'embedding qui transforment le texte en vecteurs numériques. Plusieurs options s'offrent à nous :

| Modèle | Taille | Langues | Performance | Phase |
|--------|--------|---------|-------------|-------|
| all-MiniLM-L6-v2 | 22M | Anglais (cross-lingue) | Bonne | Phase 1 (prototypage) |
| bilingual-embedding-small | 118M | Français/Anglais | Très bonne | Phase 2 |
| nomic-embed-text (Ollama) | 137M | Multilingue | Excellente | Phase 2 |

**Pour la phase 1** :
- all-MiniLM-L6-v2 est léger (22M params) et rapide, idéal pour valider le pipeline RAG sans complexité.
- Bien que non spécifiquement entraîné pour le français, ses performances cross-linguistiques sont suffisantes pour un prototype.

**Pour la phase 2 (optionnel)** :
- bilingual-embedding-small pour une meilleure couverture français/anglais.
- nomic-embed-text pour des performances multilingues optimales si nécessaire.

### **Phase 1: Embedding avec `all-MiniLM-L6-v2`**

In [4]:
# Modèle d'embedding
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 636.95it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## **Extraction Du Texte Des PDFs**

In [5]:
# extraction du texte de tous les PDFs
pdf_files = list(DATA_PART2.glob("*.pdf"))
print(f"Nombre de PDFs trouvés : {len(pdf_files)}")

documents = []

for pdf_path in pdf_files:
    print(f"Traitement de : {pdf_path.name}")
    texte_complet = ""
    
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            texte_page = page.extract_text()
            if texte_page:
                texte_complet += texte_page + " "
    
    documents.append({
        "source": pdf_path.name,
        "texte": texte_complet.strip()
    })

print(f"Extraction terminée. {len(documents)} documents traités.")

Nombre de PDFs trouvés : 6
Traitement de : Principes de vie internes.pdf
Traitement de : politiques_hotel_de_la_promenade_document.pdf
Traitement de : Convention collective - services alimentaires.pdf
Traitement de : Hébergement - Critères du service d'entretien ménager dans un hôtel 4 diamants.pdf
Traitement de : Processus - Gestion des réservations.pdf
Traitement de : Membres du comité SST.pdf
Extraction terminée. 6 documents traités.


In [6]:
# conversion en DataFrame
df_docs = pd.DataFrame(documents)
df_docs

,source,texte
0,Principes de vie internes.pdf,"PRINCIPES DE VIE INTERNES\nÀ notre équipe,\nAu..."
1,politiques_hotel_de_la_promenade_document.pdf,Manuel Opérationnel & Politiques — Hôtel De la...
2,Convention collective - services alimentaires.pdf,CONVENTION COLLECTIVE\nEntre\nL’HÔTEL DE LA PR...
3,Hébergement - Critères du service d'entretien ...,Voici les critères du service d'entretien ména...
4,Processus - Gestion des réservations.pdf,Processus Gestion des réservations (défaillant...
5,Membres du comité SST.pdf,"Comité de la santé, la sécurité et du bien-êtr..."


## **Aperçu Du Contenu Extrait**

In [7]:
# aperçu des 500 premiers caractères de chaque document
for idx, row in df_docs.iterrows():
    print(f"\n--- {row['source']} ---")
    print(row['texte'][:500] + "...")


--- Principes de vie internes.pdf ---
PRINCIPES DE VIE INTERNES
À notre équipe,
Au cœur de notre hôtel, nous croyons en l'importance de valeurs solides et d'un engagement
commun envers l'excellence. Afin de guider notre travail quotidien, nous avons élaboré des
principes de vie internes qui reflètent notre vision collective et nos attentes en tant qu’équipe de
travail.
En intégrant ces principes dans nos actions et nos interactions quotidiennes, nous sommes en
mesure de créer une expérience exceptionnelle pour notre clientèle, de dé...

--- politiques_hotel_de_la_promenade_document.pdf ---
Manuel Opérationnel & Politiques — Hôtel De la
Promenade
Localisation : Ottawa, Ontario, Canada
Version du document : 3.0 (Mis à jour le 12 janvier 2026)
Usage : Interne, Formation du personnel
Table des matières
1. Présentation & informations générales
2. Mission, vision et valeurs
3. Coordonnées et contacts rapides
4. Heures d'ouverture et politiques d'accueil (Front Desk)
5. Types de chambres, éq

## **Sauvegarde Des Textes Bruts**

In [8]:
# sauvegarde pour utilisation ultérieure
df_docs.to_parquet(DATA_PROCESSED / "documents_bruts.parquet")
print(f"Documents sauvegardés dans {DATA_PROCESSED}/documents_bruts.parquet")

Documents sauvegardés dans /home/kevin/Documents/Collège LaCité/Cours IA en Informatique (51847)/Hiver 2026/Intelligence Artificielle Générative (IFM31124-0020-H2026)/Projets/ua1/assistant-rag/data/processed/documents_bruts.parquet


## **Chunking Des Documents**

In [9]:
# paramètres de chunking
chunk_size = 512  # nombre de caractères par chunk
overlap = 50      # chevauchement entre chunks

In [10]:
chunks = []

for idx, row in df_docs.iterrows():
	texte = row['texte']
	source = row['source']
	
	# découpage en chunks avec chevauchement
	for i in range(0, len(texte), chunk_size - overlap):
		chunk = texte[i:i + chunk_size]
		if len(chunk) > 100:  # ignorer les chunks trop petits
			chunks.append({
				"source": source,
				"chunk_id": f"{source}_{i}",
				"texte": chunk,
				"start_char": i,
				"end_char": i + len(chunk)
			})

In [11]:
df_chunks = pd.DataFrame(chunks)
print(f"Nombre total de chunks créés : {len(df_chunks)}")
df_chunks.head()

Nombre total de chunks créés : 148


,source,chunk_id,texte,start_char,end_char
0,Principes de vie internes.pdf,Principes de vie internes.pdf_0,"PRINCIPES DE VIE INTERNES\nÀ notre équipe,\nAu...",0,512
1,Principes de vie internes.pdf,Principes de vie internes.pdf_462,"ptionnelle pour notre clientèle, de développer...",462,974
2,Principes de vie internes.pdf,Principes de vie internes.pdf_924,it d’équipe\nNous travaillons en collaboration...,924,1436
3,Principes de vie internes.pdf,Principes de vie internes.pdf_1386,"d'éthique et d'intégrité, en\nfaisant preuve d...",1386,1898
4,Principes de vie internes.pdf,Principes de vie internes.pdf_1848,action. Nous portons attention à leurs besoins...,1848,2360


## **Sauvegarde Des Chunks Et Des Paramètres**

In [12]:
# sauvegarde des chunks
df_chunks.to_parquet(DATA_PROCESSED / "chunks.parquet")
print(f"Chunks sauvegardés dans {DATA_PROCESSED}/chunks.parquet")

Chunks sauvegardés dans /home/kevin/Documents/Collège LaCité/Cours IA en Informatique (51847)/Hiver 2026/Intelligence Artificielle Générative (IFM31124-0020-H2026)/Projets/ua1/assistant-rag/data/processed/chunks.parquet


In [13]:
# sauvegarde des paramètres de chunking pour traçabilité
chunk_params = {
	"chunk_size": chunk_size,
	"overlap": overlap,
	"total_chunks": len(df_chunks),
	"model_embedding": "all-MiniLM-L6-v2",
	"date": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open(MODELS_RAG_DIR / "chunk_params.json", "w") as f:
	json.dump(chunk_params, f, indent=2)

print("Paramètres de chunking sauvegardés")

Paramètres de chunking sauvegardés


## **Création Des `Embeddings` Et `Index FAISS`**

In [14]:
# génération des embeddings pour chaque chunk
print("Génération des embeddings en cours...")
embeddings = embedding_model.encode(df_chunks['texte'].tolist(), show_progress_bar=True)
print(f"Embeddings générés : {embeddings.shape}")

Génération des embeddings en cours...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches: 100%|██████████| 5/5 [00:05<00:00,  1.11s/it]

Embeddings générés : (148, 384)


In [15]:
# création de l'index FAISS
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # distance L2 (euclidienne)
index.add(embeddings.astype('float32'))
print(f"Index FAISS créé avec {index.ntotal} vecteurs")

Index FAISS créé avec 148 vecteurs


## **Sauvegarde De L'Index Et Des Métadonnées**

In [16]:
# sauvegarde de l'index FAISS
faiss.write_index(index, str(MODELS_RAG_DIR / "index.faiss"))
print(f"Index sauvegardé dans {MODELS_RAG_DIR}/index.faiss")

Index sauvegardé dans /home/kevin/Documents/Collège LaCité/Cours IA en Informatique (51847)/Hiver 2026/Intelligence Artificielle Générative (IFM31124-0020-H2026)/Projets/ua1/assistant-rag/models/1-rag/index.faiss


In [17]:
# sauvegarde des chunks associés (pour retrouver le texte à partir des indices)
with open(MODELS_RAG_DIR / "chunks.pkl", "wb") as f:
    pickle.dump(df_chunks.to_dict('records'), f)
print("Métadonnées des chunks sauvegardées")

Métadonnées des chunks sauvegardées


In [18]:
# sauvegarde des paramètres du modèle
model_params = {
	"embedding_model": "all-MiniLM-L6-v2",
	"dimension": dimension,
	"total_chunks": len(df_chunks),
	"date": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open(MODELS_RAG_DIR / "model_params_all_miniLM.json", "w") as f:
	json.dump(model_params, f, indent=2)

print("Paramètres du modèle sauvegardés")

Paramètres du modèle sauvegardés


## **Test De Recherche (Retrieval)**

In [19]:
# fonction de recherche
def rechercher(question, k=3):
    # encoder la question
    question_embedding = embedding_model.encode([question])
    
    # recherche dans l'index
    distances, indices = index.search(question_embedding.astype('float32'), k)
    
    # afficher les résultats
    print(f"Question : {question}\n")
    for i, idx in enumerate(indices[0]):
        chunk = df_chunks.iloc[idx]
        print(f"Résultat {i+1} (distance: {distances[0][i]:.4f})")
        print(f"Source : {chunk['source']}")
        print(f"Extrait : {chunk['texte'][:200]}...")
        print("=" * 80)
    
    return indices[0]

In [20]:
# Test
rechercher("Quelle est la politique de congés?")

Question : Quelle est la politique de congés?

Résultat 1 (distance: 0.8631)
Source : Convention collective - services alimentaires.pdf
Extrait : qu'à trois (3) jours
de congé de deuil supplémentaires, mais ceux-ci ne sont pas rémunérés.
Page 22 21.2 Gestion des congés
a) La rémunération pour un congé de deuil est calculée en fonction du temps ...
Résultat 2 (distance: 0.9739)
Source : Convention collective - services alimentaires.pdf
Extrait : u du quart, sauf en
cas de motif légitime qui ne lui permettrait pas de respecter cette exigence.
ARTICLE 21 - CONGÉS DE DEUIL
21.1 Congés octroyés et rémunération
En cas de décès de certains membres ...
Résultat 3 (distance: 1.0283)
Source : Convention collective - services alimentaires.pdf
Extrait : n comité patronal-syndical qui se réunira au
besoin, mais pas plus d'une fois par mois, sauf entente mutuelle contraire. Ce comité aura pour
but de discuter des questions ou des problèmes d'intérêt co...


array([134, 132,  65])

## **Tests De Stratégies De Prompting**

In [21]:
# fonction de recherche avec contexte pour prompting
def rechercher_contexte(question, k=3):
  # encoder la question
  question_embedding = embedding_model.encode([question])
  
  # recherche dans l'index
  distances, indices = index.search(question_embedding.astype('float32'), k)
  
  # construire le contexte à partir des chunks trouvés
  contexte = ""
  sources = []
  
  for i, idx in enumerate(indices[0]):
    chunk = df_chunks.iloc[idx]
    contexte += f"[Extrait {i+1} - {chunk['source']}]\n{chunk['texte']}\n\n"
    sources.append(chunk['source'])
  
  return contexte, sources

#### **Question Test & Contexte**

In [22]:
# test de la fonction rechercher_contexte
question_test = "Quelle est la politique de congés?"
contexte, sources = rechercher_contexte(question_test)
print("Contexte récupéré :")
print(contexte[:500] + "...")
print(f"Sources : {set(sources)}")

Contexte récupéré :
[Extrait 1 - Convention collective - services alimentaires.pdf]
qu'à trois (3) jours
de congé de deuil supplémentaires, mais ceux-ci ne sont pas rémunérés.
Page 22 21.2 Gestion des congés
a) La rémunération pour un congé de deuil est calculée en fonction du temps prévu pour les
quarts de travail réguliers de l'employé.
b) Ces congés rémunérés ne sont pas accordés s'ils coïncident avec d'autres congés
spécifiés dans la convention.
c) L'employé désirant se prévaloir de son droit de congé de deuil,...
Sources : {'Convention collective - services alimentaires.pdf'}


## **Stratégie `Zero-Shot`**

In [23]:
def prompt_zero_shot(question, contexte):
  prompt = f"""En utilisant uniquement les extraits de documents fournis ci-dessous, réponds à la question de l'employé.

Extraits de la documentation :
{contexte}

Question de l'employé : {question}

Réponse (base-toi strictement sur les extraits fournis) :"""
  return prompt

In [24]:
# Test
prompt = prompt_zero_shot(question_test, contexte)
print(prompt)

En utilisant uniquement les extraits de documents fournis ci-dessous, réponds à la question de l'employé.

Extraits de la documentation :
[Extrait 1 - Convention collective - services alimentaires.pdf]
qu'à trois (3) jours
de congé de deuil supplémentaires, mais ceux-ci ne sont pas rémunérés.
Page 22 21.2 Gestion des congés
a) La rémunération pour un congé de deuil est calculée en fonction du temps prévu pour les
quarts de travail réguliers de l'employé.
b) Ces congés rémunérés ne sont pas accordés s'ils coïncident avec d'autres congés
spécifiés dans la convention.
c) L'employé désirant se prévaloir de son droit de congé de deuil, doit en informer son
supérieur hiérarchique dès que possible et fournir une

[Extrait 2 - Convention collective - services alimentaires.pdf]
u du quart, sauf en
cas de motif légitime qui ne lui permettrait pas de respecter cette exigence.
ARTICLE 21 - CONGÉS DE DEUIL
21.1 Congés octroyés et rémunération
En cas de décès de certains membres de la famille, les e

## **Stratégie `Few-Shot`**

In [25]:
def prompt_few_shot(question, contexte):
  exemples = """Exemple 1 :
Question : Quels sont les critères pour un hôtel 4 diamants concernant l'entretien ménager ?
Contexte : [Extrait - Critères du service d'entretien ménager dans un hôtel 4 diamants.pdf] Un préposé répond au téléphone rapidement, soit après moins de trois coups. Un préposé accueille le client avec chaleur et sincérité en l'appelant par son nom.
Réponse : Les critères incluent une réponse téléphonique en moins de trois coups, un accueil chaleureux avec appel du client par son nom, et une conclusion chaleureuse.

Exemple 2 :
Question : Qui sont les membres du comité SST ?
Contexte : [Extrait - Membres du comité SST.pdf] Le Comité de la santé, la sécurité et du bien-être des employées et employés de l'Hôtel de la Promenade a pour objectif de veiller à la prévention des accidents...
Réponse : Le comité SST veille à la prévention des accidents et à l'identification des risques. Il se réunit au moins 6 fois par an.

Maintenant, réponds à cette nouvelle question en te basant strictement sur les extraits fournis."""

  prompt = f"""{exemples}

Question : {question}
Contexte :
{contexte}
Réponse :"""
  return prompt


In [26]:
# Test
prompt = prompt_few_shot(question_test, contexte)
print(prompt)

Exemple 1 :
Question : Quels sont les critères pour un hôtel 4 diamants concernant l'entretien ménager ?
Contexte : [Extrait - Critères du service d'entretien ménager dans un hôtel 4 diamants.pdf] Un préposé répond au téléphone rapidement, soit après moins de trois coups. Un préposé accueille le client avec chaleur et sincérité en l'appelant par son nom.
Réponse : Les critères incluent une réponse téléphonique en moins de trois coups, un accueil chaleureux avec appel du client par son nom, et une conclusion chaleureuse.

Exemple 2 :
Question : Qui sont les membres du comité SST ?
Contexte : [Extrait - Membres du comité SST.pdf] Le Comité de la santé, la sécurité et du bien-être des employées et employés de l'Hôtel de la Promenade a pour objectif de veiller à la prévention des accidents...
Réponse : Le comité SST veille à la prévention des accidents et à l'identification des risques. Il se réunit au moins 6 fois par an.

Maintenant, réponds à cette nouvelle question en te basant stricte

## **Stratégie `Chain-of-Thought (CoT)`**

In [27]:
def prompt_cot(question, contexte):
  prompt = f"""En utilisant uniquement les extraits de documents fournis ci-dessous, réponds à la question de l'employé en expliquant étape par étape ton raisonnement.

Extraits de la documentation :
{contexte}

Question de l'employé : {question}

Étape 1 : Identifie les extraits pertinents pour la question.
Étape 2 : Extrais les informations spécifiques qui répondent à la question.
Étape 3 : Formule une réponse complète basée sur ces informations.

Réponse étape par étape :"""
  return prompt

# test
prompt = prompt_cot(question_test, contexte)
print(prompt)

En utilisant uniquement les extraits de documents fournis ci-dessous, réponds à la question de l'employé en expliquant étape par étape ton raisonnement.

Extraits de la documentation :
[Extrait 1 - Convention collective - services alimentaires.pdf]
qu'à trois (3) jours
de congé de deuil supplémentaires, mais ceux-ci ne sont pas rémunérés.
Page 22 21.2 Gestion des congés
a) La rémunération pour un congé de deuil est calculée en fonction du temps prévu pour les
quarts de travail réguliers de l'employé.
b) Ces congés rémunérés ne sont pas accordés s'ils coïncident avec d'autres congés
spécifiés dans la convention.
c) L'employé désirant se prévaloir de son droit de congé de deuil, doit en informer son
supérieur hiérarchique dès que possible et fournir une

[Extrait 2 - Convention collective - services alimentaires.pdf]
u du quart, sauf en
cas de motif légitime qui ne lui permettrait pas de respecter cette exigence.
ARTICLE 21 - CONGÉS DE DEUIL
21.1 Congés octroyés et rémunération
En cas de

## **Documentation Des Prompts Testés**

On documente ici les 03 stratégies de prompting évaluées : zero-shot, few-shot et chain-of-thought. Pour chacune, on présente le template utilisé, sa justification, ses limites et un exemple issu de nos tests.

In [28]:
# sauvegarde des prompts testés
prompts_testes = {
  "zero_shot": {
    "description": "Prompt simple demandant de répondre uniquement à partir du contexte fourni.",
    "template": "En utilisant uniquement les extraits de documents fournis ci-dessous, réponds à la question de l'employé.\n\nExtraits de la documentation :\n{contexte}\n\nQuestion de l'employé : {question}\n\nRéponse (base-toi strictement sur les extraits fournis) :",
    "justification": "Idéal pour des questions factuelles simples où le contexte contient directement la réponse. Testé avec la question 'Quelle est la politique de congés?' - les extraits pertinents ont bien été récupérés (Article 21 sur les congés de deuil).",
    "exemple_contexte": "congés de deuil, 5 jours rémunérés pour certains membres de la famille",
    "limite": "Peut générer des réponses trop vagues si le contexte est partiel."
  },
  "few_shot": {
    "description": "Prompt avec deux exemples de questions/réponses avant la question cible.",
    "template": "Exemple 1 : ... Exemple 2 : ... \n\nQuestion : {question}\nContexte : {contexte}\nRéponse :",
    "justification": "Utile pour guider le format de réponse attendu et pour des questions complexes nécessitant un format spécifique. Les exemples choisis (entretien ménager 4 diamants et comité SST) couvrent différents types de documentation.",
    "exemples_utilises": "entretien ménager (critères service), comité SST (composition et rôle)",
    "limite": "Peut biaiser le modèle vers le format des exemples plutôt que le contenu recherché."
  },
  "cot": {
    "description": "Prompt guidant le modèle à raisonner étape par étape.",
    "template": "En utilisant uniquement les extraits de documents fournis ci-dessous, réponds à la question de l'employé en expliquant étape par étape ton raisonnement.\n\nExtraits de la documentation :\n{contexte}\n\nQuestion de l'employé : {question}\n\nÉtape 1 : Identifie les extraits pertinents pour la question.\nÉtape 2 : Extrais les informations spécifiques qui répondent à la question.\nÉtape 3 : Formule une réponse complète basée sur ces informations.\n\nRéponse étape par étape :",
    "justification": "Recommandé pour les questions nécessitant une synthèse ou impliquant plusieurs extraits. Réduit les hallucinations en forçant le modèle à expliciter son raisonnement. Particulièrement utile quand l'information est dispersée dans plusieurs sections.",
    "cas_utilisation": "questions complexes, informations dispersées, besoin de traçabilité du raisonnement",
    "limite": "Réponses plus longues, peut être excessif pour des questions simples."
  }
}

with open(MODELS_RAG_DIR / "prompts_documentation.json", "w") as f:
  import json
  json.dump(prompts_testes, f, indent=2, ensure_ascii=False)

print("Documentation des prompts sauvegardée")

Documentation des prompts sauvegardée


## **Conversations Test (10-15 Scénarios)**

In [29]:
# liste des questions test pour différents scénarios
questions_test = [
  {
    "categorie": "Ressources humaines",
    "question": "Quels sont les congés de deuil prévus dans la convention collective?"
  },
  {
    "categorie": "Ressources humaines", 
    "question": "Combien de jours de préavis pour une démission?"
  },
  {
    "categorie": "Entretien ménager",
    "question": "Quels sont les critères pour un hôtel 4 diamants concernant le service d'entretien ménager?"
  },
  {
    "categorie": "Entretien ménager",
    "question": "Comment un préposé doit-il répondre au téléphone selon les critères 4 diamants?"
  },
  {
    "categorie": "Réservations",
    "question": "Quelle est la procédure de gestion des réservations?"
  },
  {
    "categorie": "Réservations",
    "question": "Comment gérer une surréservation?"
  },
  {
    "categorie": "Politiques internes",
    "question": "Quelle est la tenue vestimentaire requise pour les employés?"
  },
  {
    "categorie": "Politiques internes",
    "question": "Quels sont les principes de vie internes de l'hôtel?"
  },
  {
    "categorie": "Santé et sécurité",
    "question": "Qui sont les membres du comité SST?"
  },
  {
    "categorie": "Santé et sécurité",
    "question": "Que faire en cas d'accident de travail?"
  },
  {
    "categorie": "Services alimentaires",
    "question": "Quelles sont les conditions de travail prévues par la convention collective pour les services alimentaires?"
  },
  {
    "categorie": "Services alimentaires",
    "question": "Comment fonctionne le comité patronal-syndical?"
  }
]

print(f"Nombre de scénarios préparés : {len(questions_test)}")
df_questions = pd.DataFrame(questions_test)
df_questions

Nombre de scénarios préparés : 12


,categorie,question
0,Ressources humaines,Quels sont les congés de deuil prévus dans la ...
1,Ressources humaines,Combien de jours de préavis pour une démission?
2,Entretien ménager,Quels sont les critères pour un hôtel 4 diaman...
3,Entretien ménager,Comment un préposé doit-il répondre au télépho...
4,Réservations,Quelle est la procédure de gestion des réserva...
5,Réservations,Comment gérer une surréservation?
6,Politiques internes,Quelle est la tenue vestimentaire requise pour...
7,Politiques internes,Quels sont les principes de vie internes de l'...
8,Santé et sécurité,Qui sont les membres du comité SST?
9,Santé et sécurité,Que faire en cas d'accident de travail?


In [30]:
# fonction pour exécuter une conversation test complète
def tester_question(question, categorie, strategie="zero_shot"):
  print(f"---- {categorie} ----")
  print(f"Question : {question}\n")
  
  # recherche du contexte
  contexte, sources = rechercher_contexte(question)
  print(f"Sources pertinentes : {set(sources)}\n")
  
  # génération du prompt selon la stratégie choisie
  if strategie == "zero_shot":
    prompt = prompt_zero_shot(question, contexte)
  elif strategie == "few_shot":
    prompt = prompt_few_shot(question, contexte)
  else:  # cot
    prompt = prompt_cot(question, contexte)
  
  print("Prompt généré :")
  print(prompt)
  print("\n" + "#"*80 + "\n")
  
  return {
    "categorie": categorie,
    "question": question,
    "sources": list(set(sources)),
    "contexte": contexte,
    "prompt": prompt
  }

In [31]:
# Test sur une question pour vérifier
test_resultat = tester_question(
  questions_test[0]["question"], 
  questions_test[0]["categorie"],
  strategie="zero_shot"
)

---- Ressources humaines ----
Question : Quels sont les congés de deuil prévus dans la convention collective?

Sources pertinentes : {'Convention collective - services alimentaires.pdf'}

Prompt généré :
En utilisant uniquement les extraits de documents fournis ci-dessous, réponds à la question de l'employé.

Extraits de la documentation :
[Extrait 1 - Convention collective - services alimentaires.pdf]
qu'à trois (3) jours
de congé de deuil supplémentaires, mais ceux-ci ne sont pas rémunérés.
Page 22 21.2 Gestion des congés
a) La rémunération pour un congé de deuil est calculée en fonction du temps prévu pour les
quarts de travail réguliers de l'employé.
b) Ces congés rémunérés ne sont pas accordés s'ils coïncident avec d'autres congés
spécifiés dans la convention.
c) L'employé désirant se prévaloir de son droit de congé de deuil, doit en informer son
supérieur hiérarchique dès que possible et fournir une

[Extrait 2 - Convention collective - services alimentaires.pdf]
u du quart, sauf

## **Observations (Partie Retrival)**
- Extraction des PDFs réussie malgré quelques textes tronqués ("grand-pèr" au lieu de "grand-père").
- Recherche pertinente : les chunks retournés correspondent bien aux questions posées (ex: congés → article 21).
- Index de 148 chunks, suffisant pour la documentation existante.

**Choix effectués:**
- **Modèle d'embedding** : all-MiniLM-L6-v2 pour la phase 1 (léger et rapide). bilingual-embedding-small et nomic-embed-text réservés à la phase 2 si nécessaire.
- **Chunking** : 512 caractères avec chevauchement de 50.
- **Prompts** : Trois stratégies documentées (zero-shot, few-shot, CoT).
- **Sauvegarde** : Paramètres distincts par modèle (model_params_all_miniLM.json).

**Évaluation du choix d'embedding**

All-MiniLM fonctionne correctement pour l'instant. 
Les Limites identifiées:
- Texte tronqué dans certains extraits PDF.

## **Phase 2 - Embedding Avec `nomic-embed-text` (Ollama)**

**NB:** `bilingual-embedding-small` présentait quelques soucis techniques avec la configuration actuelle de l'environnement de travail.

### **Récupération de `nomic-embed-text`**

In [36]:
# Téléchargement explicite du modèle nomic-embed-text
!ollama pull nomic-embed-text

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 970aa74c0a90:   1% ▕                  ▏ 2.9 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   3% ▕                  ▏ 7.4 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   6% ▕█                 ▏  15 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   9% ▕█                 ▏  24 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:  10% ▕█                 ▏  28 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:  14% ▕██                ▏  37 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:  17% ▕███               ▏  46 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:  18% ▕███               ▏  50 MB/274 MB                  pulling manifest 
pulling 970aa74c

In [37]:
def get_embedding_nomic(text, model="nomic-embed-text"):
  """Genere un embedding pour un texte donne en utilisant Ollama."""
  try:
    response = ollama.embeddings(model=model, prompt=text)
    return response['embedding']
  except Exception as e:
    print(f"Erreur lors de la generation de l'embedding: {e}")
    return None

In [38]:
# Test sur un échantillon
test_text = "Ceci est un test d'embedding avec nomic-embed-text."
embedding = get_embedding_nomic(test_text)

if embedding:
  print(f"Embedding genere avec succes. Dimension: {len(embedding)}")
  print(f"Premieres valeurs: {embedding[:5]}")
else:
  print("Echec de la generation de l'embedding.")

Embedding genere avec succes. Dimension: 768
Premieres valeurs: [0.9572287797927856, 0.3948264420032501, -3.2659077644348145, -1.8735986948013306, 1.1235765218734741]


### **Génération des embeddings pour tous les chunks avec `nomic-embed-text`**

In [39]:
# Génération des embeddings pour tous les chunks avec nomic-embed-text
def generer_embeddings_nomic(textes, batch_size=10):
  embeddings = []
  for i in range(0, len(textes), batch_size):
    batch = textes[i:i+batch_size]
    batch_embeddings = []
    for texte in batch:
      emb = get_embedding_nomic(texte)
      if emb:
        batch_embeddings.append(emb)
    embeddings.extend(batch_embeddings)
    print(f"Progression: {min(i+batch_size, len(textes))}/{len(textes)}")
  return np.array(embeddings)

In [40]:
print("Generation des embeddings avec nomic-embed-text...")
embeddings_nomic = generer_embeddings_nomic(df_chunks['texte'].tolist())
print(f"Embeddings generes : {embeddings_nomic.shape}")

Generation des embeddings avec nomic-embed-text...
Progression: 10/148
Progression: 20/148
Progression: 30/148
Progression: 40/148
Progression: 50/148
Progression: 60/148
Progression: 70/148
Progression: 80/148
Progression: 90/148
Progression: 100/148
Progression: 110/148
Progression: 120/148
Progression: 130/148
Progression: 140/148
Progression: 148/148
Embeddings generes : (148, 768)


### **Création de l'index FAISS avec `nomic-embed-text`**

In [41]:
# Création de l'index FAISS avec nomic-embed-text
dimension_nomic = embeddings_nomic.shape[1]
index_nomic = faiss.IndexFlatL2(dimension_nomic)
index_nomic.add(embeddings_nomic.astype('float32'))
print(f"Index nomic cree avec {index_nomic.ntotal} vecteurs")

Index nomic cree avec 148 vecteurs


In [42]:
# Sauvegarde de l'index et des paramètres
faiss.write_index(index_nomic, str(MODELS_RAG_DIR / "index_nomic.faiss"))
print("Index nomic sauvegarde")

model_params_nomic = {
  "embedding_model": "nomic-embed-text (via Ollama)",
  "dimension": dimension_nomic,
  "total_chunks": len(df_chunks),
  "date": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open(MODELS_RAG_DIR / "model_params_nomic.json", "w") as f:
  import json
  json.dump(model_params_nomic, f, indent=2)

print("Parametres du modele nomic sauvegardes")

Index nomic sauvegarde
Parametres du modele nomic sauvegardes


### **Observations:**

- Modèle `nomic-embed-text` fonctionnel avec Ollama.
- Embeddings de dimension 768 (vs 384 pour all-MiniLM).
- Génération plus lente mais acceptable (148 chunks traités).
- Index FAISS créé et sauvegardé.
- Reste à comparer la pertinence des recherches avec all-MiniLM.

## **Comparaison `All-MiniLM` vs `Nomic-embed-text`**

In [43]:
# fonction de recherche avec l'index nomic
def rechercher_contexte_nomic(question, k=3):
  question_embedding = get_embedding_nomic(question)
  if question_embedding is None:
    return "", []
  
  distances, indices = index_nomic.search(np.array([question_embedding]).astype('float32'), k)
  
  contexte = ""
  sources = []
  
  for i, idx in enumerate(indices[0]):
    chunk = df_chunks.iloc[idx]
    contexte += f"[Extrait {i+1} - {chunk['source']}]\n{chunk['texte']}\n\n"
    sources.append(chunk['source'])
  
  return contexte, sources

### **Test comparatif avec la même question**

In [46]:
# Test comparatif avec la même question
question_test = "Quels sont les congés de deuil prévus dans la convention collective?"

print("--- RÉSULTATS ALL-MINILM ---")
contexte_minilm, sources_minilm = rechercher_contexte(question_test)
print(f"Sources : {set(sources_minilm)}")
print(f"Premier extrait : {contexte_minilm[:1000]}...\n")

print("--- RÉSULTATS NOMIC ---")
contexte_nomic, sources_nomic = rechercher_contexte_nomic(question_test)
print(f"Sources : {set(sources_nomic)}")
print(f"Premier extrait : {contexte_nomic[:1000]}...")

--- RÉSULTATS ALL-MINILM ---
Sources : {'Convention collective - services alimentaires.pdf'}
Premier extrait : [Extrait 1 - Convention collective - services alimentaires.pdf]
qu'à trois (3) jours
de congé de deuil supplémentaires, mais ceux-ci ne sont pas rémunérés.
Page 22 21.2 Gestion des congés
a) La rémunération pour un congé de deuil est calculée en fonction du temps prévu pour les
quarts de travail réguliers de l'employé.
b) Ces congés rémunérés ne sont pas accordés s'ils coïncident avec d'autres congés
spécifiés dans la convention.
c) L'employé désirant se prévaloir de son droit de congé de deuil, doit en informer son
supérieur hiérarchique dès que possible et fournir une

[Extrait 2 - Convention collective - services alimentaires.pdf]
u du quart, sauf en
cas de motif légitime qui ne lui permettrait pas de respecter cette exigence.
ARTICLE 21 - CONGÉS DE DEUIL
21.1 Congés octroyés et rémunération
En cas de décès de certains membres de la famille, les employés bénéficient de cong

## **Observations - Comparaison `All-MiniLM` vs `Nomic-Embed-Text`**

**Analyse des résultats avec affichage étendu (1000 caractères)** :

- **All-MiniLM** : Les 3 extraits sont pertinents et proviennent tous de l'Article 21 sur les congés de deuil. Le modèle a correctement ciblé la section appropriée de la convention collective.

- **Nomic-embed-text** : 
  - Extrait 1 : Article 2 sur la discrimination (non pertinent)
  - Extrait 2 : Article 12 sur l'ancienneté (mentionne "congés" mais dans un contexte différent)
  - Extrait 3 : Probablement similaire (non affiché)

**Conclusion** : All-MiniLM surpforme clairement nomic-embed-text pour cette requête spécifique en français. Malgré sa taille plus modeste (384 dimensions), il capture mieux le sens de la question et retrouve les passages pertinents.

**Recommandation** : On conserve `all-MiniLM` comme modèle d'embedding principal pour la suite du RAG. L'investissement dans un modèle plus gros (nomic) n'apporte pas de gain évident dans ce cas d'usage.

## **Conclusion - Assistant RAG (Parties Retrieval et Tests)**

- **06 documents** transformés en **148 chunks** (taille 512, chevauchement 50).
- **Deux modèles testés** : all-MiniLM (384 dim) et nomic-embed-text (768 dim).
- **all-MiniLM retenu** : plus rapide et plus pertinent sur notre cas test.
- **Index FAISS** créés et sauvegardés dans `models/1-rag/`.
- **12 scénarios de test** préparés pour validation.
- **3 stratégies de prompting** documentées.

**Prochaine étape** : Notebook dédié à la **génération** avec un LLM (Mistral, Llama) pour compléter l'assistant RAG.